# lexicon

> Word and phrase rules: banned vocabulary, hedges, fillers, and splices

In [ ]:
#| default_exp lexicon

In [ ]:
#| hide
from nbdev.showdoc import *

These rules match words, phrases, punctuation, and Markdown emphasis in scrubbed text. They cover banned vocabulary, hedges, fillers, splices, and consequence joins. Findings include plain replacements where available. The rules use regexes without linguistic analysis.

## Sources and related tools

[Vale](https://vale.sh) runs YAML-defined prose style rules. [write-good](https://github.com/btford/write-good) checks English prose for passive voice, weasel words, and cliches. [proselint](https://github.com/amperser/proselint) implements usage advice from writers including Strunk, White, and Garner. Its authors describe the approach in "proselint: the linting of science prose, and the science of prose linting".

Slopometer implements its own rules to produce weighted findings, `write_docs` tell numbers, character spans for edits, and a score. It does not wrap those linters or use Vale's engine or packages.

The vocabulary combines `write_docs` with entries adapted from the [GOV.UK words-to-avoid list](https://www.gov.uk/guidance/style-guide/a-to-z-of-gov-uk-style#words-to-avoid). It contains public sector information licensed under the Open Government Licence v3.0.

In [ ]:
#| export
from fastcore.utils import *
from slopometer.core import *
from slopometer.segment import *


In [ ]:
from fastcore.test import *

## The lexicon factory

`lex_rule` registers a vocabulary of words or phrases with optional replacement suggestions. With `suffix=True`, a key also matches trailing word characters. For example, "leverage" matches "leverages" and "leveraged".

In [ ]:
#| export
def lex_rule(
    name, # Rule name, as in `core.rule`
    tell, # `write_docs` tell number
    weight, # Tier the findings carry
    lex, # Vocabulary: word or phrase -> plain replacement, or None where deletion is the fix
    suffix=False, # Also match inflected forms of each key?
):
    "Build and register a phrase rule that scans for `lex` entries case-insensitively"
    tail = r'\w*' if suffix else ''
    pat = re.compile(r'\b(' + '|'.join(re.escape(k) for k in sorted(lex, key=len, reverse=True)) + r')' + tail + r'\b', re.I)
    def _f(txt): return [Finding(name, tell, m.start(), m.end(), m.group(), weight, lex.get(m.group(1).lower())) for m in pat.finditer(txt)]
    _f.__name__,_f.__doc__ = f'find_{name}',f'Scan for the {name} vocabulary'
    return rule(name, tell=tell, weight=weight, level='phrase')(_f)

## Banned vocabulary

The list combines `write_docs` vocabulary, GOV.UK entries such as "liaise" and "overarching", and marketing cliches. It flags matches regardless of part of speech or meaning. For example, "leverage" flags as both a noun and a verb. Review the context before applying a suggestion.

The `syntax` notebook implements rules for words whose treatment depends on part of speech.

In [ ]:
#| export
banned = {'utilize': 'use', 'utilise': 'use', 'leverage': 'use', 'facilitate': 'help', 'robust': 'strong',
    'comprehensive': 'complete', 'seamless': 'smooth', 'enhance': 'improve', 'streamline': None,
    'empower': None, 'foster': 'encourage', 'pivotal': None, 'a testament to': None, 'realm': None,
    'landscape': None, 'delve': None, 'myriad': None, 'plethora': None, 'paradigm': None,
    'synergy': None, 'holistic': None, 'catalyze': None, 'catalyse': None, 'juxtapose': None,
    'tapestry': None, 'embark': None, 'endeavor': None, 'endeavour': None, 'encompass': None,
    'multifaceted': None, 'elucidate': 'explain', 'nuanced': None, 'liaise': None, 'overarching': None,
    'countless': 'many', 'incentivize': None, 'incentivise': None, 'game-changer': None,
    'cutting-edge': None, 'best-in-class': None, 'deliver results': None,
    # adopted from the GOV.UK words-to-avoid list (OGL v3) and no-cliches (MIT); evidence below
    'initiate': 'start', 'tackle': 'solve', 'a clean slate': None, 'in a nutshell': None}

find_banned = lex_rule('banned', tell=None, weight=KILL, lex=banned, suffix=True)

In [ ]:
find_banned(scrub('widget leverages a robust paradigm to deliver results.'))

[[10] banned: 'leverages' -> 'use',
 [10] banned: 'robust' -> 'strong',
 [10] banned: 'paradigm',
 [10] banned: 'deliver results']

## Hedges, noting fillers, and filler transitions

These vocabularies use the same factory. The hedge list excludes the bare modals "may" and "might": "callers may pass `None`" grants permission. A word list cannot distinguish that use from hedging.

The noting-filler and transition lists implement tells 8 and 19 from `write_docs`.

In [ ]:
#| export
hedges = {'potentially': None, 'in some cases': None, 'should generally': None, 'may or may not': None,
    'one might argue': None, 'it could be argued': None, 'arguably': None, 'more or less': None,
    # weasel words adopted from write-good's weasel-words (MIT); evidence below
    'very': None, 'quite': None, 'extremely': None, 'fairly': None, 'exceedingly': None, 'remarkably': None,
    'surprisingly': None, 'interestingly': None, 'clearly': None, 'completely': None, 'largely': None,
    'mostly': None, 'relatively': None, 'usually': None, 'significantly': None, 'substantially': None,
    'huge': None, 'tiny': None, 'vast': None, 'few': None, 'several': None, 'excellent': None}
noting = {'note that': None, "it's worth noting": None, 'it is worth noting': None, 'worth mentioning': None,
    'importantly': None, 'notably': None, 'keep in mind': None, 'bear in mind': None, 'it should be noted': None}
transitions = {'furthermore': 'and', 'moreover': 'and', 'additionally': 'also', 'in conclusion': None,
    'when it comes to': None, 'in the realm of': None}

find_hedges = lex_rule('hedges', tell=7, weight=SMELL, lex=hedges)
find_noting = lex_rule('noting', tell=8, weight=SMELL, lex=noting)
find_transitions = lex_rule('transitions', tell=19, weight=SMELL, lex=transitions)

In [ ]:
test_eq(find_hedges(scrub('The result may be `None` when the key is missing.')), [])
find_hedges('This could potentially improve performance in some cases.') + \
    find_noting("It's worth noting that the cache is per-process.") + \
    find_transitions('Furthermore, the config is optional.')

[[3] hedges (tell 7, hedging): 'potentially',
 [3] hedges (tell 7, hedging): 'in some cases',
 [3] noting (tell 8, noting fillers): "It's worth noting",
 [3] transitions (tell 19, filler transitions): 'Furthermore' -> 'and']

## Splice punctuation and consequence glue

`find_splice` gives each em dash a weight of 4. Single and double hyphens, including spaced joins, do not score. Colons do not flag because they can introduce lists and examples.

`find_conseq` flags the ", so" join. The semicolon rule is in `syntax`, where the parse distinguishes clause joins from lists and citations.

In [ ]:
#| export
_splice = re.compile(r'—')
_conseq = re.compile(r', so\b')

@rule('splice', tell=1, weight=4, level='phrase')
def find_splice(txt):
    "Em dashes"
    return [Finding('splice', 1, m.start(), m.end(), m.group(), 4) for m in _splice.finditer(txt)]

@rule('conseq', tell=6, weight=SMELL, level='phrase')
def find_conseq(txt):
    "The ', so' consequence join"
    return [Finding('conseq', 6, m.start(), m.end(), m.group(), SMELL) for m in _conseq.finditer(txt)]

In [ ]:
test_eq(find_splice('Ranges like 3-5 and compound-words pass - as do spaced -- hyphens.'), [])
test_eq(find_splice('Decided, dropped, or open; merged or blocked; unowned.'), [])
splices = find_splice("It isn't just a poller - it's the liveness authority — or so it claims.") + \
    find_conseq('The cache is warm, so calls are fast.')
test_eq([f.weight for f in splices], [4, 3])
splices


[[4] splice (tell 1, splices): '—',
 [3] conseq (tell 6, consequence glue): ', so']

## Markdown emphasis

Tell 3 recommends emphasis through position rather than formatting. `find_emphasis` flags bold and italic markers in running prose.

In [ ]:
#| export
_emph = re.compile(r'\*\*[^*\n]+\*\*|(?<![\w*])\*[^*\s][^*\n]*\*(?![\w*])|(?<![\w_])__[^_\n]+__(?![\w_])')

@rule('emphasis', tell=3, weight=SMELL, level='phrase')
def find_emphasis(txt):
    "Bold and italic markers in running prose"
    return [Finding('emphasis', 3, m.start(), m.end(), m.group(), SMELL) for m in _emph.finditer(txt)]

In [ ]:
test_eq(find_emphasis('Multiplication a*b and glob patterns *.py pass.'), [])
find_emphasis("This **isn't going to cut it** and it's about *endurance*.")

[[3] emphasis (tell 3, emphasis devices): "**isn't going to cut it**",
 [3] emphasis (tell 3, emphasis devices): '*endurance*']

## Mined vocabularies

We reviewed candidate entries from write-good's weasel words, `too-wordy`, and `no-cliches` lists, all MIT-licensed. The GOV.UK list uses OGL v3.

The original review counted matches across 5,493 assistant replies from local session transcripts. `theory4` and two clean READMEs provided a false-positive check. A candidate needed matches in the replies and none in the clean samples.

That review accepted 22 of 34 weasel words, eight wordy phrases, two of 723 cliches, and four GOV.UK entries. Cliches such as "a far cry" and "a tough row to hoe" did not occur in the sampled replies. These results describe that corpus, not every use of the words.

The evidence cell requires network access and private transcripts. It does not run in CI.

In [ ]:
#| eval: false
# The evidence run: fetch the source lists, count candidate hits on the reply corpus and the clean fixtures.
# Network and private transcripts keep this cell out of CI; rerun it when revisiting adoptions.
import httpx, re
from ghapi.skill import GhApi
def _extract(js): return [w for w in re.findall(r"'([^']+)'", js) if not w.startswith('./')]
lists = {name: _extract(httpx.get(f'https://raw.githubusercontent.com/{o}/{r}/HEAD/{p}').text)
    for name, o, r, p in [('weasel', 'btford', 'weasel-words', 'weasel.js'),
        ('wordy', 'duereg', 'too-wordy', 'wordPhrases.js'), ('cliche', 'duereg', 'no-cliches', 'cliches.js')]}
def hits(phrase, txt): return len(re.findall(r'\b' + re.escape(phrase.lower()) + r'\b', txt))
# corpus_txt: turn-final reply texts from the llmsurgery mirror; clean_txt: theory4 + clean READMEs
sorted(((w, hits(w, corpus_txt), hits(w, clean_txt)) for w in lists['weasel']), key=lambda r: -r[1])[:12]

In [ ]:
#| export
wordy = {'all of': 'all', 'additional': 'more', 'a number of': 'some', 'along the lines of': 'like',
    'already existing': 'existing', 'adjacent to': 'next to', 'anticipate': 'expect',
    'in order to': 'to', 'going forward': 'from now on'}

find_wordy = lex_rule('wordy', tell=None, weight=SMELL, lex=wordy)

In [ ]:
test_eq(find_hedges(scrub('`slopometer` scores each reply in about 30ms.')), [])
find_hedges('This is very fast and usually works.') + find_wordy('In order to run all of the tests, allow additional time.')

[[3] hedges (tell 7, hedging): 'very',
 [3] hedges (tell 7, hedging): 'usually',
 [3] wordy: 'In order to' -> 'to',
 [3] wordy: 'all of' -> 'all',
 [3] wordy: 'additional' -> 'more']

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()